In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/bench/checkpoints/pre_cell_4.pickle

In [ ]:
%%cudf.pandas.profile
### cell 4 ###

def create_dataframe_of_counts_16(dataframe, column, rename_index, rename_column, return_percentages=False):
    # Drop the first row to match the original slicing
    dataframe = dataframe.drop(index=[dataframe.index[0]])
    # Compute counts directly on GPU without explicit column indexing
    df_counts = dataframe.value_counts(subset=[column]).reset_index()
    # Rename columns
    df_counts = df_counts.rename({column: rename_index, 'count': rename_column}, axis='columns')
    if return_percentages:
        total = df_counts[rename_column].sum()
        df_counts = df_counts.assign(**{rename_column: df_counts[rename_column] * 100 / total})
    return df_counts

percentages_per_country_df = create_dataframe_of_counts_16(
    responses_df_2022_cell10,
    'In which country do you currently reside?',
    'country',
    '% of respondents',
    return_percentages=False
)
percentages_per_country_df.info()